# Generative Adversarial Attack Suite for CNN Classifiers

**Purpose:** Extend the adversarial robustness benchmark beyond gradient-based perturbations (FGSM, PGD, DeepFool).  
This notebook evaluates **semantic** vulnerabilities — feature reliance and representation bias — of VGG19, ResNet50, and DenseNet121.

**Attack Types:**
1. Latent Interpolation Attack (representation drift)
2. Style Transfer Attack — AdaIN (texture bias test)
3. Prototype Shift Attack (class-centroid manipulation)
4. Diffusion-like Noise Attack (semantic denoising)

**Key Constraints:**
- Inference only — no training or fine-tuning.
- Uses the deterministic frozen test split identical to FGSM/PGD/DeepFool.
- Results saved as `genai_results.json`, `genai_per_class_heatmap.csv`, `semantic_confusion_matrix.npy`.

## 1. Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple
from collections import defaultdict

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.vgg19 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from scipy.ndimage import median_filter
from scipy.ndimage import gaussian_filter

# Dark theme for paper-quality figures
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': 'black', 'axes.facecolor': 'black',
    'savefig.facecolor': 'black', 'text.color': 'white',
    'axes.labelcolor': 'white', 'xtick.color': 'white',
    'ytick.color': 'white', 'legend.facecolor': 'black',
    'legend.edgecolor': 'white',
})

print(f"TensorFlow: {tf.__version__}")
print(f"Eager execution: {tf.executing_eagerly()}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

## 2. Configuration

In [ ]:
# ════════════════════════════════════════════════════════════════
# CONFIGURATION  (matches training & attack notebooks exactly)
# ════════════════════════════════════════════════════════════════

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1

# Paths (relative to this notebook inside Model Training/)
CHECKPOINT_DIR = Path('checkpoints')
DATA_DIR = Path('caltech101_data')
SPLIT_FILE = Path('frozen_split_indices.json')
BASELINES_FILE = Path('clean_baselines/clean_baselines.json')
RESULTS_DIR = Path('genai_results')
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

MODEL_NAMES = ['VGG19', 'ResNet50', 'DenseNet121']

PREPROCESS_FNS = {
    'VGG19': vgg_preprocess,
    'ResNet50': resnet_preprocess,
    'DenseNet121': densenet_preprocess,
}
PREPROCESS_MODE = {
    'VGG19': 'caffe',
    'ResNet50': 'caffe',
    'DenseNet121': 'torch',
}

# Attack hyperparameters
INTERP_ALPHAS = [0.1, 0.2, 0.3, 0.4]
PROTO_LAMBDAS = [0.2, 0.4, 0.6]
DIFFUSION_STEPS = 5
DIFFUSION_SIGMA = 0.05  # Gaussian noise std per step

# Feature inversion optimisation
INVERSION_STEPS = 200
INVERSION_LR = 0.01

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Configuration loaded.")
print(f"Models: {MODEL_NAMES}")
print(f"Interpolation alphas: {INTERP_ALPHAS}")
print(f"Prototype lambdas: {PROTO_LAMBDAS}")

## 3. Load Frozen Test Split

In [ ]:
# ════════════════════════════════════════════════════════════════
# REPRODUCE EXACT TEST SPLIT from training notebook
# ════════════════════════════════════════════════════════════════

def find_image_root(base_path: Path) -> Path:
    """Find 101_ObjectCategories directory."""
    for obj_cat in base_path.rglob('101_ObjectCategories'):
        if obj_cat.is_dir():
            subdirs = [d for d in obj_cat.iterdir() if d.is_dir() and d.name != '__MACOSX']
            if len(subdirs) > 10:
                return obj_cat
    raise FileNotFoundError(f"101_ObjectCategories not found under {base_path}")

IMAGE_ROOT = find_image_root(DATA_DIR)

# Get class names (same logic as training notebook)
exclude_dirs = {'__MACOSX', '.DS_Store', 'BACKGROUND_Google', '__pycache__'}
CLASS_NAMES = sorted([
    d.name for d in IMAGE_ROOT.iterdir()
    if d.is_dir() and d.name not in exclude_dirs and not d.name.startswith('.')
])
NUM_CLASSES = len(CLASS_NAMES)
class_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
idx_to_class = {idx: name for name, idx in class_to_idx.items()}

# Collect all paths deterministically (sorted)
all_paths, all_labels = [], []
for class_name in CLASS_NAMES:
    class_dir = IMAGE_ROOT / class_name
    for img_path in sorted(class_dir.iterdir()):
        if img_path.is_file() and img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.gif', '.bmp']:
            all_paths.append(str(img_path))
            all_labels.append(class_to_idx[class_name])
all_paths = np.array(all_paths)
all_labels = np.array(all_labels)

# Load frozen indices
assert SPLIT_FILE.exists(), f"Frozen split file not found: {SPLIT_FILE}"
with open(SPLIT_FILE, 'r') as f:
    saved = json.load(f)
indices = np.array(saved['indices'])
assert saved['seed'] == SEED, f"Seed mismatch: file={saved['seed']}, expected={SEED}"

all_paths = all_paths[indices]
all_labels = all_labels[indices]

train_size = int(TRAIN_SPLIT * len(all_paths))
val_size = int(VAL_SPLIT * len(all_paths))

test_paths = all_paths[train_size + val_size:]
test_labels = all_labels[train_size + val_size:]

print(f"Image root: {IMAGE_ROOT}")
print(f"Classes: {NUM_CLASSES}")
print(f"Test samples: {len(test_paths)}")
print(f"Frozen split seed: {saved['seed']}")

# Cross-check with baselines
with open(BASELINES_FILE, 'r') as f:
    baselines = json.load(f)
assert baselines['test_samples'] == len(test_paths), \
    f"Test count mismatch: baselines={baselines['test_samples']}, computed={len(test_paths)}"
print(f"\u2705 Test split matches clean_baselines.json ({len(test_paths)} samples)")

## 4. Build Deterministic Test Dataset

In [ ]:
def build_raw_test_dataset(paths: np.ndarray, labels: np.ndarray,
                           img_size: Tuple[int, int],
                           batch_size: int) -> tf.data.Dataset:
    """Build deterministic test dataset yielding RAW [0,1] images + labels.
    No augmentation, no shuffle, no model-specific preprocessing."""
    def load_image(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

raw_test_ds = build_raw_test_dataset(test_paths, test_labels, IMG_SIZE, BATCH_SIZE)

# Verify first batch
for imgs, labs in raw_test_ds.take(1):
    print(f"Batch shape: {imgs.shape}, dtype: {imgs.dtype}")
    print(f"Pixel range: [{imgs.numpy().min():.3f}, {imgs.numpy().max():.3f}]")
    print(f"Labels shape: {labs.shape}, sample: {labs[:5].numpy()}")
print("\n\u2705 Raw test dataset built (deterministic, [0,1], no augmentation)")

## 5. Preprocessing Adapters

In [ ]:
def preprocess_for_model(images_01: tf.Tensor, model_name: str) -> tf.Tensor:
    """Convert [0,1] images to model-specific input format.
    VGG19/ResNet50 (caffe): [0,1] -> [0,255] -> BGR -> mean subtraction
    DenseNet121 (torch):    [0,1] -> normalize(mean, std)
    """
    if model_name in ('VGG19', 'ResNet50'):
        x = images_01 * 255.0
        x = x[..., ::-1]  # RGB -> BGR
        mean = tf.constant([103.939, 116.779, 123.68], dtype=tf.float32)
        x = x - mean
    else:  # DenseNet121 — torch mode
        mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
        std = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)
        x = (images_01 - mean) / std
    return x

print("\u2705 Preprocessing adapters defined.")

## 6. Load Trained Models

In [ ]:
models: Dict[str, keras.Model] = {}

for name in MODEL_NAMES:
    ckpt = CHECKPOINT_DIR / f"{name}_best.h5"
    assert ckpt.exists(), f"Checkpoint not found: {ckpt}"
    model = keras.models.load_model(ckpt)
    models[name] = model
    params = model.count_params()
    clean_acc = baselines['models'][name]['accuracy']
    print(f"{name}: {params:,} params | clean acc = {clean_acc*100:.2f}% | loaded from {ckpt}")

print(f"\n\u2705 All {len(models)} models loaded. Inference only — no training.")

## 7. Build Feature Extractors (Penultimate Layer)

For each model, create a sub-model that outputs the penultimate-layer feature vector (before the final classification head).  
This is used by the Latent Interpolation and Prototype Shift attacks.

In [ ]:
feature_extractors: Dict[str, keras.Model] = {}

for name, model in models.items():
    # The penultimate layer is the GlobalAveragePooling / Flatten output
    # before the final Dense(101) classification head.
    # For typical fine-tuned models: input -> base -> GAP -> Dense(101)
    # We want the GAP output.
    penultimate_layer = model.layers[-2]  # layer before final Dense
    feat_model = keras.Model(
        inputs=model.input,
        outputs=penultimate_layer.output,
        name=f"{name}_features"
    )
    feature_extractors[name] = feat_model
    print(f"{name}: feature dim = {penultimate_layer.output_shape[-1]} "
          f"(layer: {penultimate_layer.name})")

print(f"\n\u2705 Feature extractors built for all {len(feature_extractors)} models.")

## 8. Extract Test Set Features & Identify Correctly Classified Samples

For each model:
- Extract penultimate features for every test image.
- Record the predicted label.
- Only correctly classified images are used for attacks.

In [ ]:
# Storage: features, predictions, correct masks per model
test_features: Dict[str, np.ndarray] = {}
test_preds: Dict[str, np.ndarray] = {}
correct_masks: Dict[str, np.ndarray] = {}

for name in MODEL_NAMES:
    print(f"\nExtracting features for {name}...")
    feat_list, pred_list = [], []
    for imgs_01, labs in raw_test_ds:
        preprocessed = preprocess_for_model(imgs_01, name)
        feats = feature_extractors[name](preprocessed, training=False)
        logits = models[name](preprocessed, training=False)
        preds = tf.argmax(logits, axis=-1)
        feat_list.append(feats.numpy())
        pred_list.append(preds.numpy())

    test_features[name] = np.concatenate(feat_list, axis=0)
    test_preds[name] = np.concatenate(pred_list, axis=0)
    correct_masks[name] = (test_preds[name] == test_labels)

    n_correct = correct_masks[name].sum()
    print(f"  {name}: {n_correct}/{len(test_labels)} correctly classified "
          f"({n_correct/len(test_labels)*100:.1f}%)")
    print(f"  Feature shape: {test_features[name].shape}")

print(f"\n\u2705 Feature extraction complete for all models.")

## 9. Compute Class Prototypes

For each model, compute the mean feature vector per class using only correctly classified test samples.

$\mu_c = \frac{1}{|S_c|} \sum_{x \in S_c} f(x)$

In [ ]:
class_prototypes: Dict[str, np.ndarray] = {}  # {model_name: (num_classes, feat_dim)}

for name in MODEL_NAMES:
    feat_dim = test_features[name].shape[1]
    prototypes = np.zeros((NUM_CLASSES, feat_dim), dtype=np.float32)
    counts = np.zeros(NUM_CLASSES, dtype=np.int32)

    mask = correct_masks[name]
    for i in range(len(test_labels)):
        if mask[i]:
            c = test_labels[i]
            prototypes[c] += test_features[name][i]
            counts[c] += 1

    # Avoid division by zero for classes with no correct samples
    safe_counts = np.maximum(counts, 1)
    prototypes = prototypes / safe_counts[:, None]
    class_prototypes[name] = prototypes

    non_empty = (counts > 0).sum()
    print(f"{name}: prototypes computed for {non_empty}/{NUM_CLASSES} classes "
          f"(feat_dim={feat_dim})")

print(f"\n\u2705 Class prototypes computed for all models.")

## 10. Feature Inversion Utility

Given a target feature vector, optimise a [0,1] image to match that feature representation.  
Used by the Latent Interpolation and Prototype Shift attacks.

$\hat{x} = \arg\min_{x \in [0,1]} \| f(x) - z_{target} \|_2^2 + \lambda_{TV} \cdot TV(x)$

In [ ]:
def invert_features(target_features: np.ndarray,
                    init_image_01: np.ndarray,
                    model_name: str,
                    steps: int = INVERSION_STEPS,
                    lr: float = INVERSION_LR,
                    tv_weight: float = 1e-4) -> np.ndarray:
    """Optimise image pixels to match target feature vector.

    Args:
        target_features: (feat_dim,) target feature vector.
        init_image_01: (H, W, 3) initial image in [0,1].
        model_name: which model's features to match.
        steps: optimisation steps.
        lr: learning rate.
        tv_weight: total variation regularisation weight.

    Returns:
        Optimised image in [0,1] as numpy array (H, W, 3).
    """
    target_t = tf.constant(target_features[None, :], dtype=tf.float32)

    # Initialise from the original image
    img_var = tf.Variable(init_image_01[None, ...], dtype=tf.float32)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

    feat_model = feature_extractors[model_name]

    for _ in range(steps):
        with tf.GradientTape() as tape:
            preprocessed = preprocess_for_model(img_var, model_name)
            feats = feat_model(preprocessed, training=False)
            # Feature matching loss
            feat_loss = tf.reduce_mean(tf.square(feats - target_t))
            # Total variation for smoothness
            tv_loss = tf.reduce_sum(tf.image.total_variation(img_var))
            loss = feat_loss + tv_weight * tv_loss

        grads = tape.gradient(loss, [img_var])
        optimizer.apply_gradients(zip(grads, [img_var]))
        # Clamp to [0, 1]
        img_var.assign(tf.clip_by_value(img_var, 0.0, 1.0))

    return img_var.numpy()[0]

print("\u2705 Feature inversion utility defined.")

---
# PART 2 — ATTACK IMPLEMENTATIONS

---

## 11. Attack 1 — Latent Interpolation Attack

**Idea:** Simulate representation drift by interpolating in latent space.

For each correctly classified sample $x$ with label $A$:
1. Find its nearest neighbour $x_{target}$ from a different class $B$ in feature space.
2. Interpolate: $z = (1-\alpha) f(x) + \alpha f(x_{target})$ for $\alpha \in \{0.1, 0.2, 0.3, 0.4\}$.
3. Reconstruct image via feature inversion.
4. Measure classification change vs $\alpha$.

In [ ]:
def find_nearest_different_class(features: np.ndarray, labels: np.ndarray,
                                  query_idx: int, mask: np.ndarray) -> int:
    """Find nearest neighbour from a different class among correctly classified samples."""
    query_feat = features[query_idx]
    query_label = labels[query_idx]

    best_dist = np.inf
    best_idx = -1
    for j in range(len(features)):
        if j == query_idx or not mask[j] or labels[j] == query_label:
            continue
        dist = np.sum((features[j] - query_feat) ** 2)
        if dist < best_dist:
            best_dist = dist
            best_idx = j
    return best_idx


def load_single_image(path: str, img_size: Tuple[int, int] = IMG_SIZE) -> np.ndarray:
    """Load a single image as [0,1] numpy array, shape (H, W, 3)."""
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, img_size)
    img = tf.cast(img, tf.float32) / 255.0
    return img.numpy()


def predict_single(image_01: np.ndarray, model_name: str) -> Tuple[int, float]:
    """Predict class and confidence for a single [0,1] image."""
    preprocessed = preprocess_for_model(
        tf.constant(image_01[None, ...], dtype=tf.float32), model_name
    )
    logits = models[model_name](preprocessed, training=False)
    probs = tf.nn.softmax(logits, axis=-1).numpy()[0]
    pred = int(np.argmax(probs))
    conf = float(probs[pred])
    return pred, conf

print("\u2705 Helper functions defined.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# ATTACK 1: LATENT INTERPOLATION ATTACK
# ════════════════════════════════════════════════════════════════

MAX_SAMPLES_PER_ATTACK = 100  # Cap for computational tractability

latent_interp_results: Dict[str, dict] = {}

for name in MODEL_NAMES:
    print(f"\n{'='*60}")
    print(f"Latent Interpolation Attack — {name}")
    print(f"{'='*60}")

    mask = correct_masks[name]
    correct_indices = np.where(mask)[0]
    # Subsample if needed
    np.random.seed(SEED)
    if len(correct_indices) > MAX_SAMPLES_PER_ATTACK:
        sample_indices = np.random.choice(correct_indices, MAX_SAMPLES_PER_ATTACK, replace=False)
    else:
        sample_indices = correct_indices
    sample_indices = np.sort(sample_indices)

    alpha_results = {alpha: {'fooled': 0, 'total': 0,
                             'class_changes': []} for alpha in INTERP_ALPHAS}

    for count, idx in enumerate(sample_indices):
        if count % 20 == 0:
            print(f"  Processing {count}/{len(sample_indices)}...")

        # Find nearest different-class neighbour
        nn_idx = find_nearest_different_class(
            test_features[name], test_labels, idx, mask
        )
        if nn_idx == -1:
            continue

        orig_feat = test_features[name][idx]
        target_feat = test_features[name][nn_idx]
        true_label = int(test_labels[idx])
        target_label = int(test_labels[nn_idx])
        orig_image = load_single_image(test_paths[idx])

        for alpha in INTERP_ALPHAS:
            # Interpolate in feature space
            z_interp = (1 - alpha) * orig_feat + alpha * target_feat
            # Reconstruct via feature inversion
            recon = invert_features(z_interp, orig_image, name)
            # Classify reconstructed
            pred, conf = predict_single(recon, name)

            alpha_results[alpha]['total'] += 1
            if pred != true_label:
                alpha_results[alpha]['fooled'] += 1
            alpha_results[alpha]['class_changes'].append({
                'idx': int(idx),
                'true': true_label,
                'target': target_label,
                'pred': pred,
                'conf': conf,
                'fooled': pred != true_label
            })

    # Summarise
    model_summary = {}
    for alpha in INTERP_ALPHAS:
        r = alpha_results[alpha]
        rate = r['fooled'] / r['total'] if r['total'] > 0 else 0.0
        model_summary[str(alpha)] = {
            'fooling_rate': rate,
            'fooled': r['fooled'],
            'total': r['total'],
            'class_changes': r['class_changes']
        }
        print(f"  alpha={alpha}: fooling rate = {rate*100:.1f}% "
              f"({r['fooled']}/{r['total']})")

    latent_interp_results[name] = model_summary

print(f"\n\u2705 Latent Interpolation Attack complete.")

## 12. Attack 2 — Style Transfer Attack (Texture Bias Test)

**Hypothesis:** CNNs rely on texture over shape. If we transfer texture from a different class, the prediction should flip.

**Method:** Adaptive Instance Normalisation (AdaIN):
$$\text{AdaIN}(x, y) = \sigma(y) \left( \frac{x - \mu(x)}{\sigma(x)} \right) + \mu(y)$$

Content = original image, Style = random image from a different class.

In [ ]:
def adain_style_transfer(content: np.ndarray, style: np.ndarray,
                          model_name: str, alpha: float = 1.0) -> np.ndarray:
    """Apply AdaIN style transfer in feature space, then invert.

    AdaIN(content, style) = sigma(style) * (content - mu(content)) / sigma(content) + mu(style)

    We work at the feature level: extract intermediate features from a
    mid-level layer, apply AdaIN, then optimise the image to match.
    """
    # Use the feature extractor (penultimate layer) for AdaIN
    content_t = tf.constant(content[None, ...], dtype=tf.float32)
    style_t = tf.constant(style[None, ...], dtype=tf.float32)

    content_pre = preprocess_for_model(content_t, model_name)
    style_pre = preprocess_for_model(style_t, model_name)

    content_feat = feature_extractors[model_name](content_pre, training=False).numpy()[0]
    style_feat = feature_extractors[model_name](style_pre, training=False).numpy()[0]

    # AdaIN in feature space (1-D vector: channel-level stats)
    c_mean, c_std = content_feat.mean(), content_feat.std() + 1e-8
    s_mean, s_std = style_feat.mean(), style_feat.std() + 1e-8

    adain_feat = s_std * (content_feat - c_mean) / c_std + s_mean
    # Blend with content features
    blended_feat = alpha * adain_feat + (1 - alpha) * content_feat

    # Invert blended features back to pixel space
    stylized = invert_features(blended_feat, content, model_name,
                               steps=150, lr=0.01)
    return stylized

print("\u2705 AdaIN style transfer function defined.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# ATTACK 2: STYLE TRANSFER ATTACK
# ════════════════════════════════════════════════════════════════

style_transfer_results: Dict[str, dict] = {}

for name in MODEL_NAMES:
    print(f"\n{'='*60}")
    print(f"Style Transfer Attack — {name}")
    print(f"{'='*60}")

    mask = correct_masks[name]
    correct_indices = np.where(mask)[0]
    np.random.seed(SEED)
    if len(correct_indices) > MAX_SAMPLES_PER_ATTACK:
        sample_indices = np.random.choice(correct_indices, MAX_SAMPLES_PER_ATTACK, replace=False)
    else:
        sample_indices = correct_indices
    sample_indices = np.sort(sample_indices)

    fooled_count = 0
    total_count = 0
    class_pair_fooling = defaultdict(lambda: {'fooled': 0, 'total': 0})
    details = []

    for count, idx in enumerate(sample_indices):
        if count % 20 == 0:
            print(f"  Processing {count}/{len(sample_indices)}...")

        true_label = int(test_labels[idx])
        content_img = load_single_image(test_paths[idx])

        # Pick random style image from a different class
        diff_class_indices = np.where(
            (test_labels != true_label) & mask
        )[0]
        if len(diff_class_indices) == 0:
            continue
        style_idx = np.random.choice(diff_class_indices)
        style_label = int(test_labels[style_idx])
        style_img = load_single_image(test_paths[style_idx])

        # Apply style transfer
        stylized = adain_style_transfer(content_img, style_img, name)

        # Classify
        pred, conf = predict_single(stylized, name)

        total_count += 1
        is_fooled = pred != true_label
        if is_fooled:
            fooled_count += 1

        pair_key = f"{idx_to_class[true_label]}->{idx_to_class[style_label]}"
        class_pair_fooling[pair_key]['total'] += 1
        if is_fooled:
            class_pair_fooling[pair_key]['fooled'] += 1

        details.append({
            'idx': int(idx),
            'true': true_label,
            'style_class': style_label,
            'pred': pred,
            'conf': conf,
            'fooled': is_fooled
        })

    rate = fooled_count / total_count if total_count > 0 else 0.0
    style_transfer_results[name] = {
        'fooling_rate': rate,
        'fooled': fooled_count,
        'total': total_count,
        'class_pair_fooling': dict(class_pair_fooling),
        'details': details
    }
    print(f"  Fooling rate: {rate*100:.1f}% ({fooled_count}/{total_count})")

print(f"\n\u2705 Style Transfer Attack complete.")

## 13. Attack 3 — Prototype Shift Attack

For sample $x$ from class $A$, shift its feature representation toward class $B$'s prototype:

$$f_{adv} = f(x) + \lambda (\mu_B - \mu_A)$$

$\lambda \in \{0.2, 0.4, 0.6\}$

Reconstruct image and measure the minimum $\lambda$ that causes label change.

In [ ]:
# ════════════════════════════════════════════════════════════════
# ATTACK 3: PROTOTYPE SHIFT ATTACK
# ════════════════════════════════════════════════════════════════

proto_shift_results: Dict[str, dict] = {}

for name in MODEL_NAMES:
    print(f"\n{'='*60}")
    print(f"Prototype Shift Attack — {name}")
    print(f"{'='*60}")

    mask = correct_masks[name]
    correct_indices = np.where(mask)[0]
    np.random.seed(SEED)
    if len(correct_indices) > MAX_SAMPLES_PER_ATTACK:
        sample_indices = np.random.choice(correct_indices, MAX_SAMPLES_PER_ATTACK, replace=False)
    else:
        sample_indices = correct_indices
    sample_indices = np.sort(sample_indices)

    lambda_results = {lam: {'fooled': 0, 'total': 0,
                            'details': []} for lam in PROTO_LAMBDAS}
    min_lambda_for_change = []

    for count, idx in enumerate(sample_indices):
        if count % 20 == 0:
            print(f"  Processing {count}/{len(sample_indices)}...")

        true_label = int(test_labels[idx])
        orig_feat = test_features[name][idx]
        orig_img = load_single_image(test_paths[idx])

        # Pick a random target class different from true
        target_classes = [c for c in range(NUM_CLASSES) if c != true_label]
        target_class = np.random.choice(target_classes)

        proto_A = class_prototypes[name][true_label]
        proto_B = class_prototypes[name][target_class]
        shift_dir = proto_B - proto_A

        sample_min_lambda = None

        for lam in PROTO_LAMBDAS:
            f_adv = orig_feat + lam * shift_dir
            recon = invert_features(f_adv, orig_img, name)
            pred, conf = predict_single(recon, name)

            lambda_results[lam]['total'] += 1
            is_fooled = pred != true_label
            if is_fooled:
                lambda_results[lam]['fooled'] += 1
                if sample_min_lambda is None:
                    sample_min_lambda = lam

            lambda_results[lam]['details'].append({
                'idx': int(idx),
                'true': true_label,
                'target_class': int(target_class),
                'pred': pred,
                'conf': conf,
                'fooled': is_fooled
            })

        if sample_min_lambda is not None:
            min_lambda_for_change.append(sample_min_lambda)

    # Summarise
    model_summary = {}
    for lam in PROTO_LAMBDAS:
        r = lambda_results[lam]
        rate = r['fooled'] / r['total'] if r['total'] > 0 else 0.0
        model_summary[str(lam)] = {
            'fooling_rate': rate,
            'fooled': r['fooled'],
            'total': r['total'],
            'details': r['details']
        }
        print(f"  lambda={lam}: fooling rate = {rate*100:.1f}% "
              f"({r['fooled']}/{r['total']})")

    avg_min_lambda = np.mean(min_lambda_for_change) if min_lambda_for_change else None
    model_summary['min_lambda_for_change'] = {
        'mean': float(avg_min_lambda) if avg_min_lambda else None,
        'values': min_lambda_for_change,
        'count': len(min_lambda_for_change)
    }
    if avg_min_lambda:
        print(f"  Mean minimum lambda for label change: {avg_min_lambda:.3f} "
              f"(n={len(min_lambda_for_change)})")

    proto_shift_results[name] = model_summary

print(f"\n\u2705 Prototype Shift Attack complete.")

## 14. Attack 4 — Diffusion-like Noise Attack (Semantic Denoise)

Iteratively:
1. Add Gaussian noise: $x_{noisy} = x + \mathcal{N}(0, \sigma^2)$
2. Denoise using median filter + Gaussian blur

This simulates generative prior projection.  
**Goal:** See if model predictions change while the image remains recognisable to humans.

In [ ]:
def diffusion_denoise_attack(image_01: np.ndarray,
                              steps: int = DIFFUSION_STEPS,
                              sigma: float = DIFFUSION_SIGMA,
                              median_size: int = 3,
                              blur_sigma: float = 1.0) -> List[np.ndarray]:
    """Apply iterative noise-denoise cycles.

    Returns list of images at each step (including original at step 0).
    """
    trajectory = [image_01.copy()]
    x = image_01.copy()
    for _ in range(steps):
        # Add Gaussian noise
        noise = np.random.randn(*x.shape).astype(np.float32) * sigma
        x = x + noise
        x = np.clip(x, 0.0, 1.0)
        # Denoise: median filter per channel + Gaussian blur
        for ch in range(3):
            x[:, :, ch] = median_filter(x[:, :, ch], size=median_size)
        x = gaussian_filter(x, sigma=blur_sigma)
        x = np.clip(x, 0.0, 1.0).astype(np.float32)
        trajectory.append(x.copy())
    return trajectory

print("\u2705 Diffusion-like noise attack function defined.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# ATTACK 4: DIFFUSION-LIKE NOISE ATTACK
# ════════════════════════════════════════════════════════════════

diffusion_results: Dict[str, dict] = {}

for name in MODEL_NAMES:
    print(f"\n{'='*60}")
    print(f"Diffusion-like Noise Attack — {name}")
    print(f"{'='*60}")

    mask = correct_masks[name]
    correct_indices = np.where(mask)[0]
    np.random.seed(SEED)
    if len(correct_indices) > MAX_SAMPLES_PER_ATTACK:
        sample_indices = np.random.choice(correct_indices, MAX_SAMPLES_PER_ATTACK, replace=False)
    else:
        sample_indices = correct_indices
    sample_indices = np.sort(sample_indices)

    step_results = {step: {'fooled': 0, 'total': 0} for step in range(DIFFUSION_STEPS + 1)}
    details = []

    for count, idx in enumerate(sample_indices):
        if count % 20 == 0:
            print(f"  Processing {count}/{len(sample_indices)}...")

        true_label = int(test_labels[idx])
        orig_img = load_single_image(test_paths[idx])

        # Run diffusion trajectory
        trajectory = diffusion_denoise_attack(orig_img)

        sample_detail = {'idx': int(idx), 'true': true_label, 'steps': []}
        for step_i, img_step in enumerate(trajectory):
            pred, conf = predict_single(img_step, name)
            is_fooled = pred != true_label
            step_results[step_i]['total'] += 1
            if is_fooled:
                step_results[step_i]['fooled'] += 1
            sample_detail['steps'].append({
                'step': step_i,
                'pred': pred,
                'conf': conf,
                'fooled': is_fooled
            })
        details.append(sample_detail)

    # Summarise
    model_summary = {'per_step': {}, 'details': details}
    for step in range(DIFFUSION_STEPS + 1):
        r = step_results[step]
        rate = r['fooled'] / r['total'] if r['total'] > 0 else 0.0
        model_summary['per_step'][step] = {
            'fooling_rate': rate,
            'fooled': r['fooled'],
            'total': r['total']
        }
        label = "(clean baseline)" if step == 0 else ""
        print(f"  Step {step}: fooling rate = {rate*100:.1f}% "
              f"({r['fooled']}/{r['total']}) {label}")

    diffusion_results[name] = model_summary

print(f"\n\u2705 Diffusion-like Noise Attack complete.")

---

# PART 3 — METRICS

---

## 15. Compute Aggregate Metrics

For each model, compute:
1. **Semantic Fooling Rate** — overall fraction of semantic attacks that cause misclassification.
2. **Minimum Semantic Distance** — minimum perturbation strength (α, λ, step) to flip the label.
3. **Class-to-Class Confusion Graph** — which class pairs are most confused under semantic attack.
4. **Texture Sensitivity Score** — fooling rate of the style transfer attack.
5. **Feature Drift Required for Misclassification** — from the prototype shift attack.

In [ ]:
# ════════════════════════════════════════════════════════════════
# AGGREGATE METRICS
# ════════════════════════════════════════════════════════════════

aggregate_metrics: Dict[str, dict] = {}

for name in MODEL_NAMES:
    print(f"\n{'='*60}")
    print(f"Metrics — {name}")
    print(f"{'='*60}")

    # --- 1. Semantic Fooling Rate (average across all attacks) ---
    sfr_values = []

    # Latent interp: use strongest alpha
    best_alpha = str(max(INTERP_ALPHAS))
    li_rate = latent_interp_results[name][best_alpha]['fooling_rate']
    sfr_values.append(li_rate)

    # Style transfer
    st_rate = style_transfer_results[name]['fooling_rate']
    sfr_values.append(st_rate)

    # Prototype shift: use strongest lambda
    best_lam = str(max(PROTO_LAMBDAS))
    ps_rate = proto_shift_results[name][best_lam]['fooling_rate']
    sfr_values.append(ps_rate)

    # Diffusion: use final step
    dn_rate = diffusion_results[name]['per_step'][DIFFUSION_STEPS]['fooling_rate']
    sfr_values.append(dn_rate)

    semantic_fooling_rate = float(np.mean(sfr_values))
    print(f"  1. Semantic Fooling Rate (avg): {semantic_fooling_rate*100:.1f}%")
    print(f"     Latent Interp (alpha={best_alpha}): {li_rate*100:.1f}%")
    print(f"     Style Transfer: {st_rate*100:.1f}%")
    print(f"     Proto Shift (lambda={best_lam}): {ps_rate*100:.1f}%")
    print(f"     Diffusion (step={DIFFUSION_STEPS}): {dn_rate*100:.1f}%")

    # --- 2. Minimum Semantic Distance ---
    # Latent interp: minimum alpha causing >0% fooling
    min_alpha = None
    for a in INTERP_ALPHAS:
        if latent_interp_results[name][str(a)]['fooling_rate'] > 0:
            min_alpha = a
            break

    # Proto shift: mean minimum lambda
    proto_min_lam = proto_shift_results[name].get('min_lambda_for_change', {})
    mean_min_lambda = proto_min_lam.get('mean', None)

    print(f"  2. Min Semantic Distance:")
    print(f"     Min alpha for any fooling: {min_alpha}")
    print(f"     Mean min lambda for label flip: {mean_min_lambda}")

    # --- 3. Class-to-Class Confusion Graph ---
    confusion_matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int32)

    # From latent interpolation (strongest alpha)
    for entry in latent_interp_results[name][best_alpha]['class_changes']:
        if entry['fooled']:
            confusion_matrix[entry['true'], entry['pred']] += 1

    # From style transfer
    for entry in style_transfer_results[name]['details']:
        if entry['fooled']:
            confusion_matrix[entry['true'], entry['pred']] += 1

    # From prototype shift (strongest lambda)
    for entry in proto_shift_results[name][best_lam]['details']:
        if entry['fooled']:
            confusion_matrix[entry['true'], entry['pred']] += 1

    # From diffusion (final step)
    for sample in diffusion_results[name]['details']:
        final = sample['steps'][-1]
        if final['fooled']:
            confusion_matrix[sample['true'], final['pred']] += 1

    total_confusions = confusion_matrix.sum()
    print(f"  3. Confusion matrix: {total_confusions} total misclassifications")

    # --- 4. Texture Sensitivity Score ---
    texture_sensitivity = st_rate
    print(f"  4. Texture Sensitivity Score: {texture_sensitivity*100:.1f}%")

    # --- 5. Feature Drift for Misclassification ---
    feat_drift = mean_min_lambda
    print(f"  5. Feature Drift Required (mean min lambda): {feat_drift}")

    aggregate_metrics[name] = {
        'semantic_fooling_rate': semantic_fooling_rate,
        'per_attack_fooling_rates': {
            'latent_interpolation': li_rate,
            'style_transfer': st_rate,
            'prototype_shift': ps_rate,
            'diffusion_denoise': dn_rate
        },
        'min_semantic_distance': {
            'min_alpha_for_fooling': min_alpha,
            'mean_min_lambda_for_flip': mean_min_lambda
        },
        'confusion_matrix_total': int(total_confusions),
        'texture_sensitivity_score': texture_sensitivity,
        'feature_drift_for_misclassification': feat_drift
    }

    # Save per-model confusion matrix
    np.save(RESULTS_DIR / f'{name}_semantic_confusion_matrix.npy', confusion_matrix)

print(f"\n\u2705 All metrics computed.")

## 16. Per-Class Vulnerability Heatmap

In [ ]:
# ════════════════════════════════════════════════════════════════
# PER-CLASS FOOLING HEATMAP
# ════════════════════════════════════════════════════════════════

heatmap_rows = []

for name in MODEL_NAMES:
    class_fooled = defaultdict(lambda: {'total': 0, 'fooled': 0})

    # Aggregate all attacks
    # Latent interpolation (strongest alpha)
    best_alpha = str(max(INTERP_ALPHAS))
    for entry in latent_interp_results[name][best_alpha]['class_changes']:
        cls = entry['true']
        class_fooled[cls]['total'] += 1
        if entry['fooled']:
            class_fooled[cls]['fooled'] += 1

    # Style transfer
    for entry in style_transfer_results[name]['details']:
        cls = entry['true']
        class_fooled[cls]['total'] += 1
        if entry['fooled']:
            class_fooled[cls]['fooled'] += 1

    # Prototype shift (strongest lambda)
    best_lam = str(max(PROTO_LAMBDAS))
    for entry in proto_shift_results[name][best_lam]['details']:
        cls = entry['true']
        class_fooled[cls]['total'] += 1
        if entry['fooled']:
            class_fooled[cls]['fooled'] += 1

    # Diffusion (final step)
    for sample in diffusion_results[name]['details']:
        cls = sample['true']
        final = sample['steps'][-1]
        class_fooled[cls]['total'] += 1
        if final['fooled']:
            class_fooled[cls]['fooled'] += 1

    for cls_idx in sorted(class_fooled.keys()):
        d = class_fooled[cls_idx]
        rate = d['fooled'] / d['total'] if d['total'] > 0 else 0.0
        heatmap_rows.append({
            'model': name,
            'class_idx': cls_idx,
            'class_name': idx_to_class[cls_idx],
            'total': d['total'],
            'fooled': d['fooled'],
            'fooling_rate': rate
        })

df_heatmap = pd.DataFrame(heatmap_rows)
heatmap_csv = RESULTS_DIR / 'genai_per_class_heatmap.csv'
df_heatmap.to_csv(heatmap_csv, index=False)

print(f"Per-class heatmap saved to {heatmap_csv}")
print(f"Shape: {df_heatmap.shape}")
print(f"\nTop 10 most vulnerable classes (across models):")
print(df_heatmap.sort_values('fooling_rate', ascending=False).head(10)[
    ['model', 'class_name', 'fooling_rate', 'fooled', 'total']
].to_string(index=False))

## 17. Visualise Per-Class Heatmap

In [ ]:
# ════════════════════════════════════════════════════════════════
# HEATMAP FIGURE — Top 20 most vulnerable classes
# ════════════════════════════════════════════════════════════════

pivot = df_heatmap.pivot_table(
    index='class_name', columns='model', values='fooling_rate'
)

# Select top 20 by mean fooling rate across models
pivot['mean_rate'] = pivot.mean(axis=1)
top_classes = pivot.nlargest(20, 'mean_rate').drop(columns='mean_rate')

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(
    top_classes,
    annot=True, fmt='.2f', cmap='YlOrRd',
    linewidths=0.5, linecolor='#333333',
    cbar_kws={'label': 'Semantic Fooling Rate'},
    ax=ax
)
ax.set_title('Top 20 Most Vulnerable Classes — Generative Attacks',
             fontsize=14, pad=15)
ax.set_ylabel('Class')
ax.set_xlabel('Model')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'per_class_vulnerability_heatmap.png', dpi=200)
plt.show()
print(f"\u2705 Heatmap saved.")

## 18. Confusion Matrix Visualisation

In [ ]:
# ════════════════════════════════════════════════════════════════
# SEMANTIC CONFUSION MATRICES
# ════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for ax, name in zip(axes, MODEL_NAMES):
    cm = np.load(RESULTS_DIR / f'{name}_semantic_confusion_matrix.npy')
    # Show only non-zero region for readability
    row_mask = cm.sum(axis=1) > 0
    col_mask = cm.sum(axis=0) > 0
    combined_mask = row_mask | col_mask
    if combined_mask.sum() > 0:
        cm_sub = cm[np.ix_(combined_mask, combined_mask)]
        sub_names = [CLASS_NAMES[i] for i in range(NUM_CLASSES) if combined_mask[i]]
    else:
        cm_sub = cm[:20, :20]
        sub_names = CLASS_NAMES[:20]

    # Limit to max 25 for readability
    if len(sub_names) > 25:
        top_idx = np.argsort(cm_sub.sum(axis=1) + cm_sub.sum(axis=0))[-25:]
        cm_sub = cm_sub[np.ix_(top_idx, top_idx)]
        sub_names = [sub_names[i] for i in top_idx]

    sns.heatmap(
        cm_sub, xticklabels=sub_names, yticklabels=sub_names,
        cmap='Reds', linewidths=0.3, linecolor='#333',
        ax=ax, square=True
    )
    ax.set_title(f'{name}', fontsize=13)
    ax.tick_params(labelsize=6)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Semantic Attack Confusion Matrices', fontsize=15, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'semantic_confusion_matrices.png', dpi=200,
            bbox_inches='tight')
plt.show()
print("\u2705 Confusion matrix figures saved.")

## 19. Attack Comparison Bar Chart

In [ ]:
# ════════════════════════════════════════════════════════════════
# ATTACK COMPARISON — GROUPED BAR CHART
# ════════════════════════════════════════════════════════════════

attack_names = ['Latent Interp.', 'Style Transfer', 'Proto Shift', 'Diffusion']
x = np.arange(len(attack_names))
width = 0.22
colors = {'VGG19': '#e74c3c', 'ResNet50': '#3498db', 'DenseNet121': '#2ecc71'}

fig, ax = plt.subplots(figsize=(10, 6))

for i, name in enumerate(MODEL_NAMES):
    rates = [
        aggregate_metrics[name]['per_attack_fooling_rates']['latent_interpolation'],
        aggregate_metrics[name]['per_attack_fooling_rates']['style_transfer'],
        aggregate_metrics[name]['per_attack_fooling_rates']['prototype_shift'],
        aggregate_metrics[name]['per_attack_fooling_rates']['diffusion_denoise'],
    ]
    bars = ax.bar(x + i * width, [r * 100 for r in rates], width,
                  label=name, color=colors[name], edgecolor='white',
                  linewidth=0.5)
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{rate*100:.1f}', ha='center', va='bottom', fontsize=8,
                color='white')

ax.set_xlabel('Attack Type', fontsize=12)
ax.set_ylabel('Fooling Rate (%)', fontsize=12)
ax.set_title('Generative Attack Fooling Rates by Model', fontsize=14)
ax.set_xticks(x + width)
ax.set_xticklabels(attack_names)
ax.legend()
ax.set_ylim(0, 100)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'attack_comparison_bar.png', dpi=200)
plt.show()
print("\u2705 Attack comparison chart saved.")

## 20. Latent Interpolation — Fooling Rate vs Alpha

In [ ]:
# ════════════════════════════════════════════════════════════════
# FOOLING RATE vs ALPHA (LATENT INTERPOLATION)
# ════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5))
markers = {'VGG19': 'o', 'ResNet50': 's', 'DenseNet121': 'D'}

for name in MODEL_NAMES:
    rates = [latent_interp_results[name][str(a)]['fooling_rate'] * 100
             for a in INTERP_ALPHAS]
    ax.plot(INTERP_ALPHAS, rates, marker=markers[name], label=name,
            color=colors[name], linewidth=2, markersize=8)

ax.set_xlabel(r'Interpolation $\alpha$', fontsize=12)
ax.set_ylabel('Fooling Rate (%)', fontsize=12)
ax.set_title('Latent Interpolation: Fooling Rate vs Alpha', fontsize=14)
ax.legend()
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'latent_interp_alpha_curve.png', dpi=200)
plt.show()
print("\u2705 Latent interpolation alpha curve saved.")

## 21. Diffusion Attack — Fooling Rate vs Step

In [ ]:
# ════════════════════════════════════════════════════════════════
# FOOLING RATE vs DIFFUSION STEP
# ════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5))
steps_range = list(range(DIFFUSION_STEPS + 1))

for name in MODEL_NAMES:
    rates = [diffusion_results[name]['per_step'][s]['fooling_rate'] * 100
             for s in steps_range]
    ax.plot(steps_range, rates, marker=markers[name], label=name,
            color=colors[name], linewidth=2, markersize=8)

ax.set_xlabel('Diffusion Step', fontsize=12)
ax.set_ylabel('Fooling Rate (%)', fontsize=12)
ax.set_title('Diffusion-like Noise Attack: Fooling Rate vs Step', fontsize=14)
ax.legend()
ax.set_ylim(0, 100)
ax.set_xticks(steps_range)
ax.grid(alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'diffusion_step_curve.png', dpi=200)
plt.show()
print("\u2705 Diffusion step curve saved.")

## 22. Prototype Shift — Fooling Rate vs Lambda

In [ ]:
# ════════════════════════════════════════════════════════════════
# FOOLING RATE vs LAMBDA (PROTOTYPE SHIFT)
# ════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5))

for name in MODEL_NAMES:
    rates = [proto_shift_results[name][str(l)]['fooling_rate'] * 100
             for l in PROTO_LAMBDAS]
    ax.plot(PROTO_LAMBDAS, rates, marker=markers[name], label=name,
            color=colors[name], linewidth=2, markersize=8)

ax.set_xlabel(r'Shift strength $\lambda$', fontsize=12)
ax.set_ylabel('Fooling Rate (%)', fontsize=12)
ax.set_title('Prototype Shift: Fooling Rate vs Lambda', fontsize=14)
ax.legend()
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'proto_shift_lambda_curve.png', dpi=200)
plt.show()
print("\u2705 Prototype shift lambda curve saved.")

---

# PART 4 — SAVE RESULTS

---

## 23. Save All Results

In [ ]:
# ════════════════════════════════════════════════════════════════
# SAVE  genai_results.json
# ════════════════════════════════════════════════════════════════

def make_serializable(obj):
    """Recursively convert numpy types for JSON serialisation."""
    if isinstance(obj, dict):
        return {str(k): make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [make_serializable(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, np.bool_):
        return bool(obj)
    return obj


full_results = {
    'metadata': {
        'seed': SEED,
        'test_samples': len(test_paths),
        'num_classes': NUM_CLASSES,
        'models': MODEL_NAMES,
        'max_samples_per_attack': MAX_SAMPLES_PER_ATTACK,
        'interp_alphas': INTERP_ALPHAS,
        'proto_lambdas': PROTO_LAMBDAS,
        'diffusion_steps': DIFFUSION_STEPS,
        'diffusion_sigma': DIFFUSION_SIGMA,
        'inversion_steps': INVERSION_STEPS,
        'inversion_lr': INVERSION_LR
    },
    'aggregate_metrics': aggregate_metrics,
    'latent_interpolation': latent_interp_results,
    'style_transfer': style_transfer_results,
    'prototype_shift': proto_shift_results,
    'diffusion_denoise': diffusion_results
}

json_path = RESULTS_DIR / 'genai_results.json'
with open(json_path, 'w') as f:
    json.dump(make_serializable(full_results), f, indent=2)

print(f"\u2705 Full results saved to {json_path}")
print(f"   File size: {json_path.stat().st_size / 1024:.1f} KB")

## 24. Save Combined Semantic Confusion Matrix

In [ ]:
# ════════════════════════════════════════════════════════════════
# COMBINED CONFUSION MATRIX  (sum over all models)
# ════════════════════════════════════════════════════════════════

combined_cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int32)
for name in MODEL_NAMES:
    cm = np.load(RESULTS_DIR / f'{name}_semantic_confusion_matrix.npy')
    combined_cm += cm

np.save(RESULTS_DIR / 'semantic_confusion_matrix.npy', combined_cm)
print(f"\u2705 Combined semantic confusion matrix saved.")
print(f"   Shape: {combined_cm.shape}")
print(f"   Total misclassifications: {combined_cm.sum()}")
print(f"   Non-zero entries: {(combined_cm > 0).sum()}")

## 25. Summary Table

In [ ]:
# ════════════════════════════════════════════════════════════════
# FINAL SUMMARY TABLE
# ════════════════════════════════════════════════════════════════

summary_rows = []
for name in MODEL_NAMES:
    m = aggregate_metrics[name]
    summary_rows.append({
        'Model': name,
        'Clean Acc (%)': f"{baselines['models'][name]['accuracy']*100:.1f}",
        'Semantic FR (%)': f"{m['semantic_fooling_rate']*100:.1f}",
        'Latent Interp FR (%)': f"{m['per_attack_fooling_rates']['latent_interpolation']*100:.1f}",
        'Style Transfer FR (%)': f"{m['per_attack_fooling_rates']['style_transfer']*100:.1f}",
        'Proto Shift FR (%)': f"{m['per_attack_fooling_rates']['prototype_shift']*100:.1f}",
        'Diffusion FR (%)': f"{m['per_attack_fooling_rates']['diffusion_denoise']*100:.1f}",
        'Texture Sensitivity': f"{m['texture_sensitivity_score']*100:.1f}",
        'Min Lambda Flip': f"{m['min_semantic_distance']['mean_min_lambda_for_flip']:.3f}"
            if m['min_semantic_distance']['mean_min_lambda_for_flip'] is not None else 'N/A',
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

# Save as CSV
df_summary.to_csv(RESULTS_DIR / 'genai_summary.csv', index=False)
print(f"\n\u2705 Summary table saved to {RESULTS_DIR / 'genai_summary.csv'}")

## 26. Verification Checks

In [ ]:
# ════════════════════════════════════════════════════════════════
# VERIFICATION
# ════════════════════════════════════════════════════════════════

print("Verification Checks")
print("=" * 60)

# 1. All output files exist
required_files = [
    RESULTS_DIR / 'genai_results.json',
    RESULTS_DIR / 'genai_per_class_heatmap.csv',
    RESULTS_DIR / 'semantic_confusion_matrix.npy',
]
for fpath in required_files:
    exists = fpath.exists()
    status = '\u2705' if exists else '\u274c'
    print(f"  {status} {fpath.name}")

# 2. JSON loads correctly
with open(RESULTS_DIR / 'genai_results.json', 'r') as f:
    loaded = json.load(f)
assert loaded['metadata']['seed'] == SEED
assert loaded['metadata']['test_samples'] == len(test_paths)
assert set(loaded['metadata']['models']) == set(MODEL_NAMES)
print("  \u2705 genai_results.json loads and validates")

# 3. Confusion matrix shape
cm_loaded = np.load(RESULTS_DIR / 'semantic_confusion_matrix.npy')
assert cm_loaded.shape == (NUM_CLASSES, NUM_CLASSES)
print(f"  \u2705 Confusion matrix shape: {cm_loaded.shape}")

# 4. Heatmap CSV has entries for all models
df_check = pd.read_csv(RESULTS_DIR / 'genai_per_class_heatmap.csv')
for name in MODEL_NAMES:
    assert name in df_check['model'].values, f"Missing model {name} in heatmap CSV"
print(f"  \u2705 Heatmap CSV has all models ({len(df_check)} rows)")

# 5. No training occurred (models are in inference mode)
print("  \u2705 Inference only — no model weights were modified")

print("\n" + "=" * 60)
print("ALL CHECKS PASSED")
print("=" * 60)

print(f"\nOutput directory: {RESULTS_DIR.resolve()}")
for f in sorted(RESULTS_DIR.rglob('*')):
    if f.is_file():
        print(f"  {f.name:45s} {f.stat().st_size/1024:8.1f} KB")